In [ ]:
import pandas as pd
import os
import numpy as np
import sys
import re
import io
import csv
import mysql.connector
import scipy as sp

import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
db = mysql.connector.connect(host="127.0.0.1", 
                             password = 'Yaroslav', 
                             user="root", 
                             database='dementia bank', 
                             auth_plugin='mysql_native_password')
cursor = db.cursor(buffered=True)

In [ ]:
file = open("cookietheft.gif", "rb")
image = file.read()

image_head = widgets.Image(
                    value=image,
                    format='gif',
                    width='300'
                    
                )

label_head = widgets.Label(
                    value='Stimulus image to which participants answered questions about',
                    style={'description_width': 'initial'}
                )

vbox_stimulus = widgets.VBox([image_head, label_head], layout=widgets.Layout(margin = '50px 250px 50px 250px'))

In [ ]:
classification = widgets.ToggleButtons(
            options=['Dementia', 'Control', 'Participant Summary', 'Category Description']
        )

In [ ]:
comparisonCategories = widgets.Dropdown(
    options=[('Analytic', 'Analytic '), ('Clout', 'Clout'), ('Authentic', 'Authentic'),
            ('Tone', 'Tone'), ('Words Per Sentence', 'WPS'), ('Six Letter Words', 'Sixltr'), ('Dictionary Words', 'Dic'),('Function Words', 'Funct'),
            ('Negate', 'Negate'), ('Verbs', 'Verb'), ('Adjectives', 'Adj'), ('Positive Emotion', 'PositiveEmo'), ('Negative Emotion', 'NegativeEmo'), ('Perception', 'Perception'), 
            ('Past Focus', 'Focuspast'), ('Present Focus', 'Focuspresent'), ('Future Focus', 'Focusfuture'), ('Swear Words', 'swear'), ('Filler','Filler')],
    description='Category:',
)

In [ ]:
searchBox = widgets.Text(
    value='',
    placeholder='Enter Field',
    description='Search Bar:',
    disabled=False
)

In [ ]:
searchButton = widgets.Button(
    value=False,
    description='Click to search',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='check'
)
searchContainer = widgets.HBox([searchBox, searchButton])

In [ ]:
patient_id = pd.read_csv("joined_liwc.csv", usecols = ['person_id'])
def buildSearch(search_string):
    try:
        if(int(searchBox.value) in patient_id['person_id']):
            querySearch = "Select * from liwc where Person_id = " + str((searchBox.value))
    except Exception as e:
        if(searchBox.value.lower() == "participant group"):
            querySearch = "select ParticipantGroup, count(*) as total_participants from liwc group by ParticipantGroup"
        elif(searchBox.value.lower() == "person_id"):
            querySearch = "select count(*) as total_participants from liwc"
        else:
            querySearch = "Select ParticipantGroup, round(avg(" + str(searchBox.value.lower()) + "),2) as average_" + (searchBox.value) + ", round(stddev("+searchBox.value+"),2) as stdDev from liwc group by ParticipantGroup"
    return querySearch

In [ ]:
searchOut = widgets.Output(layout={'border': '1px solid black'})

def on_button_click(c):
    worked = True
    searchQuery = buildSearch(searchBox.value)
    try:
        cursor.execute(searchQuery)
    except Exception as e:
        with searchOut:
            clear_output()
            print(searchBox.value + " not a searchable feild, try one of the listed terms")
        worked = False
    if(worked):
        with searchOut:
            clear_output()
            cursor.execute(searchQuery)
            worked = True
            result = cursor.fetchall() 
            num_fields = len(cursor.description)
            field_names = [i[0] for i in cursor.description]
            print(field_names)
            for x in result:
                print(x)
            
            
searchButton.on_click(on_button_click)
    

In [ ]:
#searchOut = widgets.Output(layout={'border': '1px solid black'})

#def changedSearch(change):
#    searchQuery = buildSearch(searchBox.value)
#    cursor.execute(searchQuery)
#    with searchOut:
#        clear_output()
#        cursor.execute(searchQuery)
#        result = cursor.fetchall() 
#        num_fields = len(cursor.description)
#        field_names = [i[0] for i in cursor.description]
#        print(field_names)
#        for x in result:
#            print(x)
            
#searchBox.observe(changedSearch, 'value')
#only run when searched then hit enter, or update widget type and build query method

In [ ]:
text0 = widgets.HTML(value="<h1>Dementia and Speech Exploration</h1>",layout=widgets.Layout(margin = '50px 250px 50px 250px'))
text1 = widgets.HTML(value="<h2>Dementia affects many aspects of one cognition, including speech.<h2>")
text2 = widgets.HTML(value="<h2>To see the effects of dementia on one's speech, interact with the linguistic dataset via the functions shown below<h2>")
searchText = widgets.HTML(value="<h3>This dataset categorized certain linguistic features based on descriptions of the cookie stimulus picture<h3>")
searchText2 = widgets.HTML(value="<h3>The categories are as listed: <h3>")
searchText3 = widgets.HTML(value="<h3>Person_Id (pass in a number), Analytic, Clout, Authentic, Tone, Words Per Sentence (Wps), SixLetterWords (Sixltr), Function Words (Funct), Negate, Verb, Adjectives (Adj), Positive Emotion (PositiveEmo), NegativeEmotion (NegativeEmo), Perception, Focuspast, Focuspresent, Focusfuture, Swear, and Filler<h3>")
searchText4 = widgets.HTML(value="<h3>Search one of the categories above to view how it differs between the control and dementia groups<h3>")

vbox_search = widgets.VBox([text0, text1, vbox_stimulus, text2, searchText, searchText2, searchText3, searchText4, searchContainer, searchOut])

searchText5 = widgets.HTML(value="<h3>Or select categories from the list and buttons down below to explore in more detail<h3>")
vbox_explore = widgets.VBox([searchText5, classification, comparisonCategories])

vbox_net = widgets.VBox([vbox_search, vbox_explore])


In [ ]:
def buildQuery(classification, comparisonCategories): #CHANGE PARTICIPANT SUMMARY
    if( classification == "Dementia"):
        where = "ParticipantGroup"
        sqlQuery = "select Person_id, " + comparisonCategories + " from liwc where ParticipantGroup = 'Dementia'"      
    elif(classification == "Control"):
        sqlQuery = "select Person_id, " + comparisonCategories + " from liwc where ParticipantGroup = 'Control'"
    elif(classification == "Participant Summary"):
        #cat = comparisonCategories
        #sqlQuery = "select ParticipantGroup, round(avg(" + cat + "),2) as avg_" + cat + ", round(stddev("+cat+"),2) as stdDev, count(*) as total_participants from liwc group by ParticipantGroup"
        sqlQuery = "select a.ParticipantGroup, count(a.participantGroup) as numParticipants, round(Cast(avg(b.Age) as float),2) as avg_age, round(std(b.Age),2) as stdDev_age from bio b right outer join liwc a on b.Person_id = a.Person_id group by participantGroup"
    elif(classification == "Category Description"):
        sqlQuery = "select * from descriptions where Category = '" + comparisonCategories + "'"
    return sqlQuery    

In [ ]:
out = widgets.Output(layout={'border': '1px solid black'})

def changedToggle(change):
    sqlQuery = buildQuery(classification.value, comparisonCategories.value)
    cursor.execute(sqlQuery)
    with out:
        try:
            clear_output()
            cursor.execute(sqlQuery)
            result = cursor.fetchall() 
            num_fields = len(cursor.description)
            field_names = [i[0] for i in cursor.description]
            print(field_names)
            for x in result:
                print(x)
        except ProgrammingError:
            ""
            
classification.observe(changedToggle, 'value')
comparisonCategories.observe(changedToggle, 'value')

In [ ]:
page = widgets.VBox([vbox_net, out])
display(page)

In [ ]:
!pip freeze > requirements.txt